# J2C-TST-IT: reproducible evaluation

Evaluates the Italian baselines and any new system. The XLM-R classifier uses early stopping with patience 1 on validation accuracy. An external CSV only needs `row_id` and `generated_style`, one row per official test example. Add it to `EXTRA_SUBMISSIONS` or copy it into the Italian predictions directory.

In [5]:
from pathlib import Path
import os
import json
import subprocess
import sys

def find_evaluation_dir():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        for possible in (candidate, candidate / '6-evaluation'):
            if (possible / 'evaluate_metrics.py').is_file():
                return possible.resolve()
    raise FileNotFoundError('Cannot locate 6-evaluation/evaluate_metrics.py')

EVALUATION_DIR = find_evaluation_dir()
LANGUAGE = 'IT'
RUN_DIR = EVALUATION_DIR / 'J2C-TST-IT'
PREDICTIONS_DIR = EVALUATION_DIR.parent / '5-baselines' / 'J2C-TST-IT' / 'predictions'
CLASSIFIER_DIR = RUN_DIR / 'classifier' / 'early_stopping_p1'
RESULTS_DIR = RUN_DIR / 'results'
EXTRA_SUBMISSIONS = []  # e.g. [Path('/path/to/my_model_predictions.csv')]
ALLOW_PARTIAL = False  # keep False for comparable final results
INSTALL_DEPENDENCIES = True  # modern stack only; BELA uses BELA_PYTHON below
RUN_CLASSIFIER_TRAINING = True
RUN_MODERN_METRICS = True
RUN_BELA = True
METRIC_BATCH_SIZE = 32
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
BELA_PYTHON = EVALUATION_DIR / '.venv-bela' / 'bin' / 'python'
BELA_REPO = EVALUATION_DIR / 'external' / 'BELA'
def pick_atas_cache_base():
    explicit = os.environ.get('ATAS_CACHE_BASE')
    candidates = [explicit, '/tmp', f"/tmp/{os.environ.get('USER', '')}", f"/scratch/{os.environ.get('USER', '')}", os.environ.get('SCRATCH'), os.environ.get('LOCAL_SCRATCH'), os.environ.get('TMPDIR'), str(Path.home() / '.cache')]
    for value in candidates:
        if not value:
            continue
        candidate = Path(value).expanduser()
        try:
            candidate.mkdir(parents=True, exist_ok=True)
            probe = candidate / '.j2ctst_bela_write_test'
            probe.write_text('ok', encoding='utf-8')
            probe.unlink()
            return candidate.resolve()
        except OSError:
            continue
    raise RuntimeError('No writable directory found for BELA assets.')
ATAS_CACHE_BASE = pick_atas_cache_base()
default_bela_assets = ATAS_CACHE_BASE / 'BELA-models'
BELA_ASSETS_DIR = Path(os.environ.get('J2CTST_BELA_ASSETS_DIR', default_bela_assets)).expanduser().resolve()
print('Evaluation directory:', EVALUATION_DIR)
print('Prediction directory:', PREDICTIONS_DIR)
print('ATAS-compatible cache base:', ATAS_CACHE_BASE)
print('BELA assets directory:', BELA_ASSETS_DIR)

Evaluation directory: /home/user/J2C-TST/6-evaluation
Prediction directory: /home/user/J2C-TST/5-baselines/J2C-TST-IT/predictions
ATAS-compatible cache base: /tmp
BELA assets directory: /tmp/BELA-models


In [3]:
if INSTALL_DEPENDENCIES:
    command = [sys.executable, '-m', 'pip', 'install', '-r', str(EVALUATION_DIR / 'requirements.txt')]
    print('>', subprocess.list2cmdline(command))
    subprocess.run(command, cwd=EVALUATION_DIR, check=True)
    if sys.platform.startswith('linux'):
        flash_env = os.environ.copy()
        flash_env.setdefault('MAX_JOBS', '2')
        flash_env.setdefault('NVCC_THREADS', '2')
        command = [sys.executable, '-m', 'pip', 'install', '--no-build-isolation', '-r', str(EVALUATION_DIR / 'requirements-flash-attn.txt')]
        print('>', subprocess.list2cmdline(command))
        subprocess.run(command, cwd=EVALUATION_DIR, env=flash_env, check=True)
    elif RUN_MODERN_METRICS:
        raise RuntimeError('The FlashAttention 2 profile requires a Linux CUDA environment.')
    if RUN_MODERN_METRICS:
        check = "import torch, flash_attn; assert torch.cuda.is_available(), 'CUDA unavailable'; assert torch.cuda.is_bf16_supported(), 'The GPU does not support BF16'; print(torch.cuda.get_device_name(0), 'BF16:', torch.cuda.is_bf16_supported(), 'flash-attn:', flash_attn.__version__)"
        subprocess.run([sys.executable, '-c', check], cwd=EVALUATION_DIR, check=True)
else:
    print('Modern dependency installation skipped; set INSTALL_DEPENDENCIES=True in the configuration cell for a fresh environment.')

> /opt/ice4hpc/rel-24.04/conda/bin/python -m pip install -r /home/user/J2C-TST/6-evaluation/requirements.txt
Defaulting to user installation because normal site-packages is not writeable
> /opt/ice4hpc/rel-24.04/conda/bin/python -m pip install --no-build-isolation -r /home/user/J2C-TST/6-evaluation/requirements-flash-attn.txt
Defaulting to user installation because normal site-packages is not writeable
NVIDIA L40S BF16: True flash-attn: 2.8.3


In [ ]:
if RUN_BELA:
    command = [sys.executable, 'setup_bela.py', '--environment', BELA_PYTHON.parent.parent, '--repo', BELA_REPO, '--assets-dir', BELA_ASSETS_DIR]
    print('>', subprocess.list2cmdline([str(item) for item in command]))
    subprocess.run([str(item) for item in command], cwd=EVALUATION_DIR, check=True)
    if not BELA_PYTHON.is_file():
        raise FileNotFoundError(f'BELA setup completed without creating the expected interpreter: {BELA_PYTHON}')
else:
    print('BELA setup skipped because RUN_BELA=False.')

In [6]:
def run_script(arguments, python_executable=None):
    executable = python_executable or sys.executable
    command = [str(executable), *map(str, arguments)]
    print('\n>', subprocess.list2cmdline(command))
    process_env = os.environ.copy()
    if python_executable is not None and Path(executable) == Path(BELA_PYTHON):
        process_env.pop('PYTORCH_CUDA_ALLOC_CONF', None)
    return subprocess.run(command, cwd=EVALUATION_DIR, env=process_env, check=True)

classifier_args = ['train_style_classifier.py', '--language', LANGUAGE, '--output-dir', CLASSIFIER_DIR,
                   '--early-stopping-patience', '1', '--early-stopping-threshold', '0.0', '--resume']
if not RUN_CLASSIFIER_TRAINING:
    classifier_args.extend(['--max-train-pairs', '2', '--max-validation-pairs', '2', '--max-test-pairs', '2', '--dry-run'])
run_script(classifier_args)
if RUN_MODERN_METRICS and not RUN_CLASSIFIER_TRAINING and not (CLASSIFIER_DIR / 'config.json').is_file():
    raise FileNotFoundError(f'Train the style classifier first or set RUN_CLASSIFIER_TRAINING=True: {CLASSIFIER_DIR}')


> /opt/ice4hpc/rel-24.04/conda/bin/python train_style_classifier.py --language IT --output-dir /home/user/J2C-TST/6-evaluation/J2C-TST-IT/classifier/early_stopping_p1 --early-stopping-patience 1 --early-stopping-threshold 0.0 --resume
{
  "language": "IT",
  "corpus": "/home/user/J2C-TST/4-official_corpora/IT/J2C_TST_IT.csv",
  "model_id": "FacebookAI/xlm-roberta-base",
  "train_pairs": 26015,
  "validation_pairs": 3716,
  "test_pairs": 7432,
  "labels": {
    "journalistic": 0,
    "conversational": 1
  },
  "max_length": 512,
  "epochs": 3.0,
  "early_stopping_patience": 1,
  "early_stopping_threshold": 0.0,
  "restore_callback_states_from_checkpoint": true,
  "learning_rate": 2e-05,
  "train_batch_size": 16,
  "eval_batch_size": 32,
  "gradient_accumulation_steps": 1,
  "seed": 42,
  "output_dir": "/home/user/J2C-TST/6-evaluation/J2C-TST-IT/classifier/early_stopping_p1"
}
Completed classifier already exists; skipping: /home/user/J2C-TST/6-evaluation/J2C-TST-IT/classifier/early_stop

In [8]:
prediction_files = list(PREDICTIONS_DIR.glob('*.csv')) if PREDICTIONS_DIR.exists() else []
for submission in EXTRA_SUBMISSIONS:
    submission = Path(submission).expanduser().resolve()
    if not submission.is_file():
        raise FileNotFoundError(f'Submission not found: {submission}')
    prediction_files.append(submission)
prediction_files = sorted(set(path.resolve() for path in prediction_files), key=str)
stems = [path.stem for path in prediction_files]
if len(stems) != len(set(stems)):
    raise ValueError('Submission filenames must have unique stems to avoid result collisions')
if not prediction_files:
    print('No submissions found. Expected CSV columns: row_id, generated_style')
prediction_files

[PosixPath('/home/user/J2C-TST/5-baselines/J2C-TST-IT/predictions/IT_gemma_few_shot_gpu48_bf16_fa2_es1_gemma_b2.csv'),
 PosixPath('/home/user/J2C-TST/5-baselines/J2C-TST-IT/predictions/IT_gemma_lora_gpu48_bf16_fa2_es1_gemma_b2.csv'),
 PosixPath('/home/user/J2C-TST/5-baselines/J2C-TST-IT/predictions/IT_gemma_prefix_gpu48_bf16_fa2_es1_gemma_b2.csv'),
 PosixPath('/home/user/J2C-TST/5-baselines/J2C-TST-IT/predictions/IT_llama_few_shot_gpu48_bf16_fa2_es1_b4.csv'),
 PosixPath('/home/user/J2C-TST/5-baselines/J2C-TST-IT/predictions/IT_llama_lora_gpu48_bf16_fa2_es1_b4.csv'),
 PosixPath('/home/user/J2C-TST/5-baselines/J2C-TST-IT/predictions/IT_llama_prefix_gpu48_bf16_fa2_es1_b4.csv')]

In [ ]:
for prediction in prediction_files:
    result_dir = RESULTS_DIR / prediction.stem
    args = ['evaluate_metrics.py', '--language', LANGUAGE, '--predictions', prediction,
            '--style-classifier', CLASSIFIER_DIR, '--output-dir', result_dir,
            '--batch-size', METRIC_BATCH_SIZE, '--bf16',
            '--attention-implementation', 'flash_attention_2', '--resume']
    if ALLOW_PARTIAL:
        args.append('--allow-partial')
    if not RUN_MODERN_METRICS:
        args.append('--dry-run')
    run_script(args)

In [9]:
if RUN_BELA and (BELA_PYTHON is None or not Path(BELA_PYTHON).is_file()):
    raise FileNotFoundError(f'Create the dedicated BELA environment or set BELA_PYTHON: {BELA_PYTHON}')
for prediction in prediction_files:
    result_dir = RESULTS_DIR / prediction.stem / 'entities'
    args = ['entity_faithfulness.py', '--language', LANGUAGE, '--predictions', prediction,
            '--output-dir', result_dir, '--bela-repo', BELA_REPO,
            '--checkpoint', BELA_ASSETS_DIR / 'model_e2e.ckpt',
            '--embeddings', BELA_ASSETS_DIR / 'embeddings.pt',
            '--entity-index', BELA_ASSETS_DIR / 'index.txt',
            '--mention-threshold', '0.2', '--entity-threshold', '0.05', '--resume']
    if ALLOW_PARTIAL:
        args.append('--allow-partial')
    if not RUN_BELA:
        args.append('--dry-run')
    run_script(args, BELA_PYTHON if RUN_BELA else None)


> /home/user/J2C-TST/6-evaluation/.venv-bela/bin/python entity_faithfulness.py --language IT --predictions /home/user/J2C-TST/5-baselines/J2C-TST-IT/predictions/IT_gemma_few_shot_gpu48_bf16_fa2_es1_gemma_b2.csv --output-dir /home/user/J2C-TST/6-evaluation/J2C-TST-IT/results/IT_gemma_few_shot_gpu48_bf16_fa2_es1_gemma_b2/entities --bela-repo /home/user/J2C-TST/6-evaluation/external/BELA --checkpoint /tmp/BELA-models/model_e2e.ckpt --embeddings /tmp/BELA-models/embeddings.pt --entity-index /tmp/BELA-models/index.txt --mention-threshold 0.2 --entity-threshold 0.05 --resume


/home/user/J2C-TST/6-evaluation/.venv-bela/lib/python3.9/site-packages/hydra/experimental/initialize.py:80: UserWarning: hydra.experimental.initialize_config_module() is no longer experimental. Use hydra.initialize_config_module().
  deprecation_warning(message=message)
/home/user/J2C-TST/6-evaluation/.venv-bela/lib/python3.9/site-packages/hydra/experimental/initialize.py:82: UserWarning: 
The version_base parameter is not specified.
Please specify a compatability version level, or None.
Will assume defaults for version 1.1
  self.delegate = real_initialize_config_module(
/home/user/J2C-TST/6-evaluation/.venv-bela/lib/python3.9/site-packages/hydra/experimental/compose.py:25: UserWarning: hydra.experimental.compose() is no longer experimental. Use hydra.compose()
  deprecation_warning(message=message)
/home/user/J2C-TST/6-evaluation/.venv-bela/lib/python3.9/site-packages/hydra/_internal/defaults_list.py:251: UserWarning: In 'joint_el_mel_new': Defaults list is missing `_self_`. See http

{
  "language": "IT",
  "system_id": "IT_gemma_few_shot_gpu48_bf16_fa2_es1_gemma_b2",
  "submission_sha256": "dcdecd517b90eb0ede91732b0d4ab49eebf3496769204feedf71022baf448c2d",
  "corpus": "/home/user/J2C-TST/4-official_corpora/IT/J2C_TST_IT.csv",
  "predictions": "/home/user/J2C-TST/5-baselines/J2C-TST-IT/predictions/IT_gemma_few_shot_gpu48_bf16_fa2_es1_gemma_b2.csv",
  "rows": 7432,
  "expected_test_rows": 7432,
  "complete": true,
  "allow_partial": false,
  "reuse_linked_entities": null
}


100%|██████████| 1/1 [00:00<00:00,  2.12it/s]


{
  "examples": 7432,
  "reference_comparison": {
    "precision_micro": 0.04937080720831324,
    "recall_micro": 0.34958087550450173,
    "f1_micro": 0.08652220685415707,
    "precision_macro": 0.3400057127582953,
    "recall_macro": 0.8030698972269207,
    "f1_macro": 0.2788846171521523,
    "true_positive_entities": 1126,
    "generated_entities": 22807,
    "reference_entities": 3221
  },
  "source_support": {
    "precision_micro": 0.7803744464418819,
    "precision_macro": 0.823753813793605,
    "supported_generated_entities": 17798,
    "generated_entities": 22807,
    "source_entities": 114384
  },
  "entity_coverage": {
    "source_nonempty_examples": 7367,
    "reference_nonempty_examples": 2278,
    "generated_nonempty_examples": 5317,
    "source_nonempty_rate": 0.991254036598493,
    "reference_nonempty_rate": 0.3065123789020452,
    "generated_nonempty_rate": 0.7154198062432723
  },
  "empty_set_convention": "precision=1 when generated set is empty; recall=1 when expected

...

{
  "examples": 7432,
  "reference_comparison": {
    "precision_micro": 0.15567529577134825,
    "recall_micro": 0.41260478112387455,
    "f1_micro": 0.22605885354652153,
    "precision_macro": 0.5972540253521148,
    "recall_macro": 0.8208258047567789,
    "f1_macro": 0.5044041190838815,
    "true_positive_entities": 1329,
    "generated_entities": 8537,
    "reference_entities": 3221
  },
  "source_support": {
    "precision_micro": 0.8094178282769122,
    "precision_macro": 0.8944121411192983,
    "supported_generated_entities": 6910,
    "generated_entities": 8537,
    "source_entities": 114384
  },
  "entity_coverage": {
    "source_nonempty_examples": 7367,
    "reference_nonempty_examples": 2278,
    "generated_nonempty_examples": 3785,
    "source_nonempty_rate": 0.991254036598493,
    "reference_nonempty_rate": 0.3065123789020452,
    "generated_nonempty_rate": 0.5092841765339075
  },
  "empty_set_convention": "precision=1 when generated set is empty; recall=1 when expected s

/home/user/J2C-TST/6-evaluation/.venv-bela/lib/python3.9/site-packages/hydra/experimental/initialize.py:80: UserWarning: hydra.experimental.initialize_config_module() is no longer experimental. Use hydra.initialize_config_module().
  deprecation_warning(message=message)
/home/user/J2C-TST/6-evaluation/.venv-bela/lib/python3.9/site-packages/hydra/experimental/initialize.py:82: UserWarning: 
The version_base parameter is not specified.
Please specify a compatability version level, or None.
Will assume defaults for version 1.1
  self.delegate = real_initialize_config_module(
/home/user/J2C-TST/6-evaluation/.venv-bela/lib/python3.9/site-packages/hydra/experimental/compose.py:25: UserWarning: hydra.experimental.compose() is no longer experimental. Use hydra.compose()
  deprecation_warning(message=message)
/home/user/J2C-TST/6-evaluation/.venv-bela/lib/python3.9/site-packages/hydra/_internal/defaults_list.py:251: UserWarning: In 'joint_el_mel_new': Defaults list is missing `_self_`. See http

{
  "language": "IT",
  "system_id": "IT_llama_prefix_gpu48_bf16_fa2_es1_b4",
  "submission_sha256": "13426de929a0db1b60781154a349f77f4dce3ed892ce2844eb259307f8c17d24",
  "corpus": "/home/user/J2C-TST/4-official_corpora/IT/J2C_TST_IT.csv",
  "predictions": "/home/user/J2C-TST/5-baselines/J2C-TST-IT/predictions/IT_llama_prefix_gpu48_bf16_fa2_es1_b4.csv",
  "rows": 7432,
  "expected_test_rows": 7432,
  "complete": true,
  "allow_partial": false,
  "reuse_linked_entities": null
}


100%|██████████| 1/1 [00:00<00:00,  2.14it/s]


{
  "examples": 7432,
  "reference_comparison": {
    "precision_micro": 0.2394256887854094,
    "recall_micro": 0.3831108351443651,
    "f1_micro": 0.2946865671641791,
    "precision_macro": 0.694434771887005,
    "recall_macro": 0.8146148892818696,
    "f1_macro": 0.5746247393586771,
    "true_positive_entities": 1234,
    "generated_entities": 5154,
    "reference_entities": 3221
  },
  "source_support": {
    "precision_micro": 0.7518432285603415,
    "precision_macro": 0.8911358517992002,
    "supported_generated_entities": 3875,
    "generated_entities": 5154,
    "source_entities": 114384
  },
  "entity_coverage": {
    "source_nonempty_examples": 7367,
    "reference_nonempty_examples": 2278,
    "generated_nonempty_examples": 3085,
    "source_nonempty_rate": 0.991254036598493,
    "reference_nonempty_rate": 0.3065123789020452,
    "generated_nonempty_rate": 0.4150968783638321
  },
  "empty_set_convention": "precision=1 when generated set is empty; recall=1 when expected set i

In [10]:
summaries = []
for prediction in prediction_files:
    summary_path = RESULTS_DIR / prediction.stem / 'metrics_summary.json'
    if not summary_path.exists():
        continue
    item = json.loads(summary_path.read_text(encoding='utf-8'))
    entity_path = summary_path.parent / 'entities' / 'entity_metrics_summary.json'
    if entity_path.exists():
        item['entity_faithfulness'] = json.loads(entity_path.read_text(encoding='utf-8'))
    summaries.append(item)
if summaries:
    aggregate_path = RESULTS_DIR / 'all_results.json'
    aggregate_path.parent.mkdir(parents=True, exist_ok=True)
    aggregate_path.write_text(json.dumps(summaries, ensure_ascii=False, indent=2), encoding='utf-8')
    print('Wrote', aggregate_path)
summaries

Wrote /home/user/J2C-TST/6-evaluation/J2C-TST-IT/results/all_results.json


[{'language': 'IT',
  'system_id': 'IT_gemma_few_shot_gpu48_bf16_fa2_es1_gemma_b2',
  'predictions': '/home/user/J2C-TST/5-baselines/J2C-TST-IT/predictions/IT_gemma_few_shot_gpu48_bf16_fa2_es1_gemma_b2.csv',
  'examples': 7432,
  'model_id': 'google/gemma-3-1b-it',
  'strategy': 'few_shot',
  'submission': {'system_id': 'IT_gemma_few_shot_gpu48_bf16_fa2_es1_gemma_b2',
   'sha256': 'dcdecd517b90eb0ede91732b0d4ab49eebf3496769204feedf71022baf448c2d',
   'corpus': '/home/user/J2C-TST/4-official_corpora/IT/J2C_TST_IT.csv',
   'evaluated_rows': 7432,
   'expected_test_rows': 7432,
   'complete': True,
   'allow_partial': False},
  'evaluation_config': {'bert_model': 'xlm-roberta-large',
   'perplexity_model': 'utter-project/EuroLLM-1.7B',
   'batch_size': 32,
   'perplexity_max_length': 4096,
   'attention_implementation': 'flash_attention_2',
   'bf16_required': True,
   'style_classifier': '/home/user/J2C-TST/6-evaluation/J2C-TST-IT/classifier/early_stopping_p1',
   'skip_style': False,
  